In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

# Load and preprocess reviews
df = pd.read_csv("LP IV Updated Datasets/CBOW/IMDB Dataset.csv").dropna()
sentences = []
for review in df.review:
    words = simple_preprocess(review)
    sentences.append(words)
y = LabelEncoder().fit_transform(df.sentiment)
train, test, y_train, y_test = train_test_split(sentences, y, test_size=0.2)

# Train CBOW and average word vectors for each review
cbow = Word2Vec(train, min_count=1, sg=0, epochs=20)
x_train = []
for words in train:
    vector = cbow.wv.get_mean_vector(words)
    x_train.append(vector)
x_train = np.array(x_train)

x_test = []
for words in test:
    vector = cbow.wv.get_mean_vector(words)
    x_test.append(vector)
x_test = np.array(x_test)

# Train sentiment classifier
model = Sequential([
    Input(shape=(cbow.vector_size,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=20, batch_size=128)

loss, accuracy = model.evaluate(x_test, y_test)
print("Accuracy:", accuracy * 100, "%")
